In [0]:
# 01c — Bronze: Shapes (recorridos geográficos de las líneas)
import requests
import zipfile
import io
import pandas as pd
from datetime import datetime, timezone

In [0]:
CATALOG = "gobierno_abierto"
SCHEMA_BRONZE = "a_bronze"

API_BASE = "https://gobiernoabierto.cordoba.gob.ar/api/datos-abiertos/dato"
GTFS_DATASET_ID = 3319

In [0]:
resp = requests.get(f"{API_BASE}/{GTFS_DATASET_ID}/version-dato", timeout=30)
resp.raise_for_status()
gtfs_version = resp.json()["results"][0]

resp = requests.get(f"{API_BASE}/{GTFS_DATASET_ID}/version-dato/{gtfs_version['id']}/recurso", timeout=30)
resp.raise_for_status()
gtfs_url = resp.json()["results"][0]["url"]

print(f"Versión GTFS: {gtfs_version['titulo']}")
gtfs_zip = zipfile.ZipFile(io.BytesIO(requests.get(gtfs_url, timeout=60).content))


In [0]:
shapes_pdf = pd.read_csv(gtfs_zip.open("shapes.txt"))
print(f"Columnas: {list(shapes_pdf.columns)}")
print(f"Filas: {len(shapes_pdf)}")
print(f"Shapes distintos: {shapes_pdf['shape_id'].nunique()}")
shapes_pdf.head()

In [0]:
audit = {
    "_ingested_at": datetime.now(timezone.utc),
    "_source_dataset_id": str(GTFS_DATASET_ID),
    "_source_url": gtfs_url.split("?")[0],
}
for col, val in audit.items():
    shapes_pdf[col] = val

spark.createDataFrame(shapes_pdf).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_shapes")

count = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_shapes").count()
print(f"gtfs_shapes: {count} filas escritas.")